# Tutorial 03 — Clinical Plotting & Facilitator Node Detection

**GSoC 2026 Project #39 — NeuroSim | INCF / EBRAINS**

This notebook demonstrates:
1. Computing Modal Controllability to rank facilitator nodes
2. Detecting seizure gateway nodes (Epilepsy) and craving circuit nodes (AUD)
3. UMAP projection of high-dimensional energy profiles into 2D clinical state space
4. Plotting disease trajectories in the manifold-learned state space

> *'The pipeline must identify specific Facilitator Nodes... projecting control metrics
> (like Modal Controllability) back onto the cortex using Nilearn surface plotting so
> researchers can visualize exactly which anatomical regions are gateways.'*
> — Dr. Khushbu Agarwal, Neurostars (Mar 24, 2026)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns

from neurosim.connectivity import spectral_inversion_solver, normalize_matrix
from neurosim.control.metrics import (
    modal_controllability,
    average_controllability,
    rank_facilitator_nodes,
)
from neurosim.control.energy import minimum_energy

# Optional: UMAP for manifold projection
try:
    import umap
    UMAP_AVAILABLE = True
except ImportError:
    UMAP_AVAILABLE = False
    print("WARNING: umap-learn not installed. Install with: pip install umap-learn")

rng = np.random.default_rng(seed=99)
print("NeuroSim clinical plotting modules loaded.")

## Step 1: Build Connectome & Estimate A Matrix

In [ ]:
N = 50  # parcels

raw = rng.standard_normal((N, N))
fc = (raw @ raw.T) / N
np.fill_diagonal(fc, 1.0)

A, info = spectral_inversion_solver(fc, alpha=0.1, system='discrete')
A_norm = normalize_matrix(A, system='discrete')

print(f"A matrix estimated. Spectral radius: {info['spectral_radius']:.5f}")

## Step 2: Modal vs Average Controllability

- **Modal Controllability (MC):** Nodes that can drive the network into hard-to-reach states → seizure/addiction gateways
- **Average Controllability (AC):** Nodes that can broadly influence many states → general stimulation targets

In [ ]:
mc = modal_controllability(A_norm)
ac = average_controllability(A_norm)

print(f"Modal controllability range:   [{mc.min():.4f}, {mc.max():.4f}]")
print(f"Average controllability range: [{ac.min():.4f}, {ac.max():.4f}]")

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Modal controllability bar chart
colors_mc = cm.Reds(mc / mc.max())
axes[0].bar(range(N), mc, color=colors_mc, edgecolor='none')
axes[0].set_title('Modal Controllability\n(Facilitator Nodes — Drive Hard States)',
                  fontsize=12, fontweight='bold')
axes[0].set_xlabel('Node Index'); axes[0].set_ylabel('MC Score')

# Average controllability bar chart
colors_ac = cm.Blues(ac / ac.max())
axes[1].bar(range(N), ac, color=colors_ac, edgecolor='none')
axes[1].set_title('Average Controllability\n(Broad Influence Nodes)',
                  fontsize=12, fontweight='bold')
axes[1].set_xlabel('Node Index'); axes[1].set_ylabel('AC Score')

plt.suptitle('NeuroSim Module B-2: Controllability Metrics', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('controllability_metrics.png', dpi=150, bbox_inches='tight')
plt.show()

## Step 3: Rank Facilitator Nodes — Seizure & AUD Gateways

In [ ]:
top_nodes, top_scores = rank_facilitator_nodes(A_norm, top_k=10)

print("Top 10 Facilitator Nodes (ranked by Modal Controllability):")
print("-" * 50)
for rank, (node, score) in enumerate(zip(top_nodes, top_scores), 1):
    bar = '█' * int(score / top_scores[0] * 20)
    print(f"  Rank {rank:2d} | Node {node:3d} | MC = {score:.4f}  {bar}")

# Highlight top facilitators on controllability map
fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(range(N), mc, color='lightcoral', alpha=0.6, label='All nodes')
ax.bar(top_nodes, mc[top_nodes], color='darkred', alpha=0.9, label='Top 10 Facilitators')
ax.set_title('Facilitator Node Detection (Seizure/AUD Gateways)',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Node Index')
ax.set_ylabel('Modal Controllability Score')
ax.legend()
plt.tight_layout()
plt.savefig('facilitator_nodes.png', dpi=150)
plt.show()

## Step 4: UMAP — 2D Clinical State Space Projection

Project high-dimensional control energy profiles into a 2D manifold.
Each point represents a subject's full energy profile (N-dimensional).
Clusters correspond to disease states: HC, AUD, Epilepsy.

In [ ]:
# Simulate energy profiles for 3 clinical cohorts
n_per_group = 30
B = np.eye(N)
T_horizon = 3

# Base reference state (HC centroid)
x_ref = np.zeros(N); x_ref[:N//3] = 1.0

def simulate_cohort_energies(x0_base, noise_level, n_subjects):
    """Simulate energy profiles for a cohort by adding noise to a base state."""
    energies = []
    for _ in range(n_subjects):
        x0 = x0_base + rng.normal(0, noise_level, N)
        xf = x_ref + rng.normal(0, 0.05, N)  # target = HC centroid
        E = minimum_energy(A_norm, T_horizon, B, x0, xf, system='discrete')
        energies.append(E)
    return np.array(energies)

# HC cohort: low noise around healthy baseline
x_healthy = np.zeros(N); x_healthy[:N//3] = 1.0
E_hc = simulate_cohort_energies(x_healthy, noise_level=0.1, n_subjects=n_per_group)

# AUD cohort: shifted state (middle parcels active)
x_aud = np.zeros(N); x_aud[N//3:2*N//3] = 1.0
E_aud = simulate_cohort_energies(x_aud, noise_level=0.15, n_subjects=n_per_group)

# Epilepsy cohort: different region active
x_epilepsy = np.zeros(N); x_epilepsy[2*N//3:] = 1.0
E_epilepsy = simulate_cohort_energies(x_epilepsy, noise_level=0.12, n_subjects=n_per_group)

# Stack all energy profiles
all_energies = np.vstack([E_hc, E_aud, E_epilepsy])
labels = (['HC'] * n_per_group + ['AUD'] * n_per_group + ['Epilepsy'] * n_per_group)

print(f"Energy profile matrix shape: {all_energies.shape}")
print(f"Groups: HC={n_per_group}, AUD={n_per_group}, Epilepsy={n_per_group}")

In [ ]:
if UMAP_AVAILABLE:
    # UMAP projection into 2D clinical state space
    reducer = umap.UMAP(n_components=2, n_neighbors=10, min_dist=0.3, random_state=42)
    embedding = reducer.fit_transform(all_energies)

    fig, ax = plt.subplots(figsize=(9, 7))
    palette = {'HC': 'steelblue', 'AUD': 'coral', 'Epilepsy': 'mediumseagreen'}
    markers = {'HC': 'o', 'AUD': 's', 'Epilepsy': '^'}

    for group in ['HC', 'AUD', 'Epilepsy']:
        mask = [l == group for l in labels]
        ax.scatter(
            embedding[mask, 0], embedding[mask, 1],
            c=palette[group], marker=markers[group],
            s=80, alpha=0.85, edgecolors='white', linewidths=0.5,
            label=group
        )

    ax.set_title('UMAP: 2D Clinical State Space\n(Energy profiles projected into manifold)',
                 fontsize=13, fontweight='bold')
    ax.set_xlabel('UMAP Dimension 1')
    ax.set_ylabel('UMAP Dimension 2')
    ax.legend(title='Cohort', fontsize=11)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig('umap_state_space.png', dpi=150)
    plt.show()
    print("UMAP plot saved to umap_state_space.png")
else:
    # Fallback: PCA projection if UMAP not available
    from sklearn.decomposition import PCA
    pca = PCA(n_components=2, random_state=42)
    embedding = pca.fit_transform(all_energies)

    fig, ax = plt.subplots(figsize=(9, 7))
    palette = {'HC': 'steelblue', 'AUD': 'coral', 'Epilepsy': 'mediumseagreen'}
    for group in ['HC', 'AUD', 'Epilepsy']:
        mask = [l == group for l in labels]
        ax.scatter(embedding[mask, 0], embedding[mask, 1],
                   c=palette[group], s=80, alpha=0.85, label=group)
    ax.set_title('PCA: 2D Clinical State Space (UMAP fallback)', fontsize=13)
    ax.legend(title='Cohort')
    ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% var)')
    ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% var)')
    plt.tight_layout()
    plt.savefig('pca_state_space.png', dpi=150)
    plt.show()

## Summary

This notebook demonstrated the complete NeuroSim Module C pipeline:

| Step | Method | Clinical Use |
|------|--------|--------------|
| Modal Controllability | Eigenspectrum weighting | Detect seizure/AUD gateway nodes |
| Average Controllability | Inverse eigenspectrum weighting | Find broad stimulation targets |
| UMAP Projection | Manifold learning on energy profiles | Map clinical disease trajectories |

**Next steps (GSoC implementation):**
- Map modal controllability onto cortical surface using `nilearn.plotting.plot_surf_stat_map`
- Apply to real HCP / ADNI / AUD datasets from OpenNeuro
- Validate disease trajectory separability using silhouette score
